[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_1_What_Makes_Data_Messy.ipynb)

# 08.1: What Makes Data Messy?

In module 06 you learned the basics of data cleaning: dropping null rows, filling missing values, removing duplicates, and renaming columns. Those tools handle the most obvious problems. But real datasets have subtler issues that those tools can't touch.

This notebook introduces a taxonomy of messiness: the five categories of problems that keep raw data from being analysis-ready. Each category has its own set of tools, which the rest of this module covers in depth. Think of this notebook as a map of the territory you are about to explore.

## The data

We will use the same Titanic dataset from earlier modules, but now we will look at it with a more critical eye. The question is not "what can I compute from this data?" but "what is wrong with this data that I need to fix first?"

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.head()

,Survived,Pclass,Name,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


The first thing to do with any new dataset is call `.info()`. It gives you the row count, column names, non-null counts, and dtypes all at once. That one output is often enough to spot three or four problems.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 887 entries, 0 to 886
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Survived                 887 non-null    int64  
 1   Pclass                   887 non-null    int64  
 2   Name                     887 non-null    str    
 3   Sex                      887 non-null    str    
 4   Age                      887 non-null    float64
 5   Siblings/Spouses Aboard  887 non-null    int64  
 6   Parents/Children Aboard  887 non-null    int64  
 7   Fare                     887 non-null    float64
dtypes: float64(2), int64(4), str(2)
memory usage: 55.6 KB


Scan that output and you will find several things worth fixing:

- The `Age` column has 714 non-null values out of 887 rows, which means 173 values are missing.
- The `Survived` column has dtype `int64`, but survival is logically a yes/no fact, not a number.
- The `Name` column is `object` (a string), which is expected, but it packs multiple pieces of information into one field: last name, title, and first name.
- The column headers themselves have problems: `Siblings/Spouses Aboard` uses slashes and mixed case, which makes column access awkward.

These are not all the same kind of problem. They belong to different categories and need different tools.

## Category 1: Wrong types

Every column in a DataFrame has a dtype. When the dtype does not match the logical meaning of the data, comparisons and calculations can silently produce wrong results.

Look at `Survived`. Storing it as `int64` lets you compute an average, which is useful. But what happens if a raw CSV version of this data stores it as the strings `"yes"` and `"no"` instead of `1` and `0`? Or what if a column of ages is stored as strings because one row had a typo like `"twenty-two"`? The column looks numeric in the CSV but pandas reads it as `object`, and any arithmetic fails.

In [3]:
print(df.dtypes)
print()
print("Pclass dtype:", df["Pclass"].dtype)
print("Pclass unique values:", df["Pclass"].unique())

Survived                     int64
Pclass                       int64
Name                           str
Sex                            str
Age                        float64
Siblings/Spouses Aboard      int64
Parents/Children Aboard      int64
Fare                       float64
dtype: object

Pclass dtype: int64
Pclass unique values: [3 1 2]


`Pclass` stores passenger class as the integers 1, 2, and 3. Arithmetic on this column is technically valid but semantically wrong: averaging class 1 and class 3 gives `2.0`, as if the result were a real class. `Pclass` is a categorical label, not a continuous measurement. Storing it as `int64` lets accidental arithmetic through without any warning.

The fix is the `category` dtype, or converting with `astype()`. You will learn both in notebook 08.3.

## Category 2: Missing values

You already know how to count and drop nulls from module 06. In this module we go much further. The key question is not just "how many values are missing?" but "why are they missing, and does that matter for the analysis?"

In [4]:
missing_counts = df.isnull().sum()
missing_pct = df.isnull().mean().round(3) * 100
pd.DataFrame({"missing": missing_counts, "pct": missing_pct})

,missing,pct
Survived,0,0.0
Pclass,0,0.0
Name,0,0.0
Sex,0,0.0
Age,0,0.0
Siblings/Spouses Aboard,0,0.0
Parents/Children Aboard,0,0.0
Fare,0,0.0


About 19.5% of ages are missing. That is not a small number. Simply dropping those 173 rows would bias every age-related analysis toward passengers whose ages were recorded, who may not be representative of all passengers. Module 06 showed you how to drop and fill; notebook 08.2 will show you how to choose which approach is appropriate, and introduces tools like `thresh=`, `ffill()`, and `interpolate()` that handle more complex patterns.

## Category 3: Inconsistent formatting

A column can have all its values present and correctly typed but still be unusable for grouping or filtering because the same thing is written different ways. A passenger class stored as `"1st"` in some rows and `"1"` in others will produce two separate groups in a `groupby`. Whitespace, capitalization, and punctuation can all create this kind of false uniqueness.

In [5]:
# Simulate a formatting problem: what if Sex had inconsistent capitalization?
sample = df["Sex"].head(10).copy()
sample.iloc[1] = "Female"   # capitalized
sample.iloc[3] = " male "   # extra whitespace
sample.value_counts()

Sex
male      5
female    3
Female    1
 male     1
Name: count, dtype: int64

With just two edits we went from two groups to four. A `groupby("Sex")` on this column would silently split `"female"` and `"Female"` into separate groups, and `" male "` would become its own third group. This is a real problem in CSV files created by hand or merged from multiple sources.

The fix involves the `.str` accessor, which lets you normalize text across an entire column in one line. Notebooks 08.4 and 08.5 cover this toolkit in full.

## Category 4: Structural problems

Some columns hold multiple distinct pieces of information crammed into one field. The `Name` column is the clearest example in this dataset.

In [6]:
df["Name"].head(8)

0                               Mr. Owen Harris Braund
1    Mrs. John Bradley (Florence Briggs Thayer) Cum...
2                                Miss. Laina Heikkinen
3          Mrs. Jacques Heath (Lily May Peel) Futrelle
4                              Mr. William Henry Allen
5                                      Mr. James Moran
6                               Mr. Timothy J McCarthy
7                        Master. Gosta Leonard Palsson
Name: Name, dtype: str

Each name contains a last name, a social title (Mr., Mrs., Miss., Master.), and a first name, all in one string. If you wanted to analyze survival rates by title, you could not do it directly: pandas sees `"Mr. Owen Harris Braund"` and `"Mrs. John Bradley (Florence Briggs Thayer) Cumings"` as two unique strings, not as two instances of the `Mr.` and `Mrs.` categories.

The fix is to extract the structured pieces using `str.split()` or `str.extract()` with a pattern. Notebooks 08.5 and 08.6 cover those tools.

Column headers can also have structural problems. The column name `Siblings/Spouses Aboard` has a slash and spaces, which makes it harder to reference in code (`df["Siblings/Spouses Aboard"]` works but is awkward; `df.Siblings/Spouses Aboard` does not). Renaming to snake_case is a basic step in any cleaning pipeline.

## Category 5: Date and time problems

The Titanic dataset does not have date columns, but virtually every real-world dataset does. Dates are almost always stored as strings in CSV files, and string dates cannot be sorted, filtered by range, or subtracted from one another.

Consider two strings: `"2023-01-15"` and `"2023-02-01"`. Alphabetically the first comes before the second, so sorting works here by coincidence. But `"01/15/2023"` and `"02/01/2023"` sort the same way alphabetically, while `"January 15, 2023"` and `"February 1, 2023"` sort `"F"` before `"J"`, which is the wrong order.

Converting to a proper `datetime64` dtype unlocks the `.dt` accessor and timedelta arithmetic. Notebook 08.7 covers parsing, extraction, and duration calculations.

## What each notebook covers

| Notebook | Topic | Main tools |
|---|---|---|
| 08.2 | Missing data in depth | `dropna(thresh=)`, `ffill()`, `bfill()`, `interpolate()` |
| 08.3 | Type problems and conversions | `astype()`, `pd.to_numeric()`, `pd.to_datetime()`, `category` |
| 08.4 | String cleaning: the `.str` accessor | `str.lower()`, `str.strip()`, `str.replace()`, `str.contains()` |
| 08.5 | Splitting and extracting text | `str.split(expand=True)`, `str.get()`, `str.extract()` |
| 08.6 | Regular expressions | `str.replace(regex=True)`, `str.extract()` with patterns |
| 08.7 | Dates and times | `pd.to_datetime()`, `.dt` accessor, timedelta arithmetic |
| 08.8 | A complete cleaning pipeline | all of the above combined |
| 08.9 | Exercises | practice across all topics |

## What's next

You have now seen what makes data messy: wrong types, missing values, inconsistent formatting, structural problems, and date encoding. The most common and consequential of these is missing data, which is where we start. Notebook 08.2 goes well beyond the basics from module 06 and shows you how to diagnose the pattern of missing data and choose the right strategy for dealing with it.